<a href="https://colab.research.google.com/github/YusufCahyo-lgtm/24001156-22-yct-text-analysis-gold/blob/main/complement_naive_bayes_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install nltk


In [4]:
!python -m nltk.downloader stopwords

<frozen runpy>:130: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [5]:
!pip install Sastrawi

In [6]:
!pip install scikit-learn

In [1]:
import pandas as pd
df= pd.read_csv("/content/data_transjakarta.csv")
df.head()

,Unnamed: 0,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion;
0,0,f80bb079-f257-421c-8b12-f07bb2c4aaef,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,aplikasinya sangat membantu dalam pengecekan r...,5.0,0.0,2.10.1,2026-02-23 16:31:05,NaN,NaN,2.10.1;
1,1,3361a69f-80b3-4a74-8ab0-3752e12488f1,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,tidak akurat,1.0,0.0,2.10.1,2026-02-23 11:07:44,Hai kak. Mohon maaf atas ketidaknyamanannya. K...,2026-02-23 15:26:58,2.10.1;
2,2,ba248fe8-9952-4413-925c-e133d8ccbf4f,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,memoermudah banget,5.0,0.0,NaN,2026-02-23 09:36:31,NaN,NaN,;
3,3,069167e7-f0b3-463c-884e-39a7b262b546,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,sangat bagus sekarang lebih dan pelayanan nya ...,5.0,0.0,NaN,2026-02-23 09:24:05,NaN,NaN,;
4,"4,8eb976e0-6dcc-4b60-9a40-b77c4e7ef353,Penggun...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
df.shape

(2086, 14)

In [7]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import confusion_matrix, f1_score, classification_report
from imblearn.metrics import geometric_mean_score

print(">>> Tahap 1: Memuat, Memfilter, dan Melabeli Dataset Biner...")

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# Muat file data
df = df.dropna(subset=['content', 'score'])

# PERBAIKAN FATAL: Buang rating bintang 3 dan 4 agar fokus pada sentimen polaritas tegas
df = df[~df['score'].isin([3, 4])]

# Fungsi untuk mentransforma label menjadi bentuk biner "0 dan 1"
def konversi_score_ke_biner(nilai_score):
    nilai_score = int(nilai_score)
    if nilai_score <= 4:
        return 0  # Kelas 0: Buruk (Bintang 1-4)
    else:
        return 1  # Kelas 1: Baik (Bintang 5)

df['label_target'] = df['score'].apply(konversi_score_ke_biner)
#NILAI AHIR DARI LABEL YG AKAN DI TERUSKAN PD PROSES SPLITTING DATA
label_target = df['label_target']


print("\n>>> Tahap 2: Menjalankan Text Preprocessing (Mohon Menunggu)...")
stop_words_id = set(stopwords.words('indonesian'))
stop_words_en = set(stopwords.words('english'))
stop_words_gabungan = stop_words_id.union(stop_words_en)

# Amankan kata negasi biner kritis
kata_sentimen_kritis = {"tidak", "kurang", "belum", "jangan", "bukan", "tidaklah", "no", "not"}
stop_words_gabungan = stop_words_gabungan - kata_sentimen_kritis

factory = StemmerFactory()
stemmer_id = factory.create_stemmer()

kamus_slang = {
    "jg": "juga", "bgt": "banget", "tp": "tapi", "krn": "karena", "dgn": "dengan",
    "gbs": "tidak bisa", "bs": "bisa", "dr": "dari", "yg": "yang", "aja": "saja",
    "klo": "kalau", "udh": "sudah", "blm": "belum", "ga": "tidak", "gk": "tidak",
    "gw": "saya", "sy": "saya", "lu": "kamu", "sm": "sama", "sdh": "sudah",
    "bisaa": "bisa", "baguss": "bagus", "ok": "oke", "okei": "oke", "siap": "oke",
    "knp": "kenapa", "utk": "untuk", "dlm": "dalam", "tgl": "tanggal", "syll": "saya",
    "msh": "masih", "nyaman Bgt": "nyaman banget", "muak": "benci", "top": "bagus",
    "tj": "transjakarta", "lemot": "lambat", "aplikasine": "aplikasi",
    "aplikasinya": "aplikasi", "busway": "busway", "qr": "kode qr", "qris": "qris",
    "halte": "halte", "petugas": "petugas", "bis": "bus", "tije": "transjakarta",
    "armada": "bus", "scaning": "pindai", "scan": "pindai", "ngescan": "memindai",
    "error": "eror", "eror": "eror", "bug": "rusak", "crash": "rusak",
    "loading": "lambat", "slow": "lambat", "fast": "cepat", "good": "bagus",
    "bad": "buruk", "worst": "buruk", "best": "bagus", "nice": "bagus",
    "update": "perbarui", "updatenya": "perbaruan", "download": "unduh",
    "login": "masuk", "logout": "keluar", "register": "daftar", "account": "akun",
    "interface": "tampilan", "ui": "tampilan", "ux": "kenyamanan", "user": "pengguna",
    "service": "layanan", "customer": "pelanggan", "force close": "keluar sendiri",
    "fc": "keluar sendiri", "delay": "terlambat", "time out": "waktu habis",
    "alhamdulillah": "syukur bagus", "alhamdullilah": "syukur bagus",
    "bismillah": "semoga", "insyaallah": "semoga", "inshaallah": "semoga",
    "barakallah": "bagus", "subhanallah": "heran", "astagfirullah": "buruk",
    "astaghfirullah": "buruk", "masyaallah": "bagus", "mashaallah": "bagus",
    "amin": "semoga", "aamiin": "semoga", "aminn": "semoga", "ngebug": "bug"
}

kalimat_bersih_nltk = []
for teks in df["content"]:
    teks_kecil = str(teks).lower()
    teks_tanpa_url = re.sub(r'https?://\S+|www\.\S+', '', teks_kecil)
    teks_tanpa_tanda_baca = re.sub(r'[^\w\s]', '', teks_tanpa_url)
    teks_tanpa_angka = re.sub(r'[-+]?[0-9]+', '', teks_tanpa_tanda_baca)

    tokens_nltk = word_tokenize(teks_tanpa_angka)
    kata_baku = [kamus_slang[kata] if kata in kamus_slang else kata for kata in tokens_nltk]
    tokens_bersih = [kata for kata in kata_baku if kata not in stop_words_gabungan]
    tokens_stemmed = [stemmer_id.stem(huhu) for huhu in tokens_bersih]

    kalimat_jadi = " ".join(tokens_stemmed)
    kalimat_bersih_nltk.append(kalimat_jadi)
df['teks_bersih'] = kalimat_bersih_nltk

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df["teks_bersih"],
    label_target,
    test_size=0.20,
    random_state=42,
    stratify=label_target
)

>>> Tahap 1: Memuat, Memfilter, dan Melabeli Dataset Biner...


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



>>> Tahap 2: Menjalankan Text Preprocessing (Mohon Menunggu)...


In [8]:
print("\n>>> Tahap 3: Transformasi Fitur TF-IDF...")
vektorisator = TfidfVectorizer(max_features=5000)
X_train_tfidf = vektorisator.fit_transform(X_train_raw)
X_test_tfidf = vektorisator.transform(X_test_raw)


print("\n>>> Tahap 4: Mengoptimalkan Saringan Kolom via Chi-Square (X²)...")
K_fitur_terbaik = 500
chi2_selector = SelectKBest(score_func=chi2, k=K_fitur_terbaik)
X_train = chi2_selector.fit_transform(X_train_tfidf, y_train)
X_test = chi2_selector.transform(X_test_tfidf)

print("\n>>> Tahap 5: Melatih Model Complement Naive Bayes (CNB)...")
model_nb = ComplementNB(alpha=0.1)
model_nb.fit(X_train, y_train)
print("Proses Estimasi Peluang Komplemen Selesai.")


print("\n>>> Tahap 6: Menghitung Akurasi Akhir & Uji Validitas Biner...")
y_prediksi = model_nb.predict(X_test)

print("\n=== CONFUSION MATRIX BINER (2 x 2) ===")
print(confusion_matrix(y_test, y_prediksi))

print("\n=== LAPORAN PERFORMA KLASIFIKASI ===")
print(classification_report(y_test, y_prediksi, target_names=['Cibiran', 'Pujian']))

print(f"Skor Akhir Geometric Mean (G-Mean): {geometric_mean_score(y_test, y_prediksi, average='binary') * 100:.2f}%")
print("\n==================== EKSEKUSI DATA BERHASIL ====================")



>>> Tahap 3: Transformasi Fitur TF-IDF...

>>> Tahap 4: Mengoptimalkan Saringan Kolom via Chi-Square (X²)...

>>> Tahap 5: Melatih Model Complement Naive Bayes (CNB)...
Proses Estimasi Peluang Komplemen Selesai.

>>> Tahap 6: Menghitung Akurasi Akhir & Uji Validitas Biner...

=== CONFUSION MATRIX BINER (2 x 2) ===
[[ 73   3]
 [ 37 305]]

=== LAPORAN PERFORMA KLASIFIKASI ===
              precision    recall  f1-score   support

     Cibiran       0.66      0.96      0.78        76
      Pujian       0.99      0.89      0.94       342

    accuracy                           0.90       418
   macro avg       0.83      0.93      0.86       418
weighted avg       0.93      0.90      0.91       418

Skor Akhir Geometric Mean (G-Mean): 92.55%

==================== EKSEKUSI DATA BERHASIL ====================
